In [ ]:
%%sql -r dataframe_1
USE DATABASE ABC_ANALYTICS;
USE SCHEMA ANALYSES;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
    SELECT 
        driver_id,
        total_offers,
        OFFER_ACCEPTANCE_RATE    AS acceptance_rate_pct,
       BOOKING_COMPLETION_RATE  AS completion_rate_pct,
        DRIVER_ABORT_RATE       AS abort_rate_pct
    FROM vw_driver_kpi_summary
""").to_pandas()

df.columns = df.columns.str.lower()

# ── histograms side by side ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Driver KPI Distribution', fontsize=13, fontweight='bold')

metrics = [
    ('acceptance_rate_pct', 'Offer Acceptance Rate %', '#0EA5E9'),
    ('completion_rate_pct', 'Booking Completion Rate %', '#10B981'),
    ('abort_rate_pct',      'Driver Abort Rate %',       '#EF4444'),
]

for ax, (col, title, color) in zip(axes, metrics):
    ax.hist(df[col], bins=20, color=color, edgecolor='white', linewidth=0.5)
    ax.axvline(df[col].mean(), color='black', linewidth=1.5,
               linestyle='--', label=f"Avg: {df[col].mean():.1f}%")
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('%')
    ax.set_ylabel('Driver Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('driver_kpis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── summary ─────────────────────────────────────────
print(f"Total drivers       : {len(df):,}")
print(f"Avg Acceptance Rate : {df['acceptance_rate_pct'].mean():.1f}%")
print(f"Avg Completion Rate : {df['completion_rate_pct'].mean():.1f}%")
print(f"Avg Abort Rate      : {df['abort_rate_pct'].mean():.1f}%")

In [ ]:
%%sql -r dataframe_2
-- segment drivers by abort rate
WITH segmented AS (
    SELECT
        driver_id,
        CASE WHEN AVG(DRIVER_ABORT_RATE) * 100 >= 30 
             THEN 'High Abort' 
             ELSE 'Reliable' 
        END                                         AS segment,
        AVG(BOOKING_COMPLETION_RATE) * 100          AS avg_completion_rate,
        AVG(AVG_EST_REVENUE_PER_TRIP)                   AS AVG_EST_REVENUE_PER_TRIP,
        SUM(driver_abort_count)                     AS total_aborts
    FROM marts.AGG_DRIVER_ACTIVITY
    WHERE total_bookings > 0
    GROUP BY driver_id
)

SELECT
    segment,
    COUNT(DISTINCT driver_id)               AS driver_count,
    ROUND(AVG(avg_completion_rate), 1)      AS avg_completion_rate,
    ROUND(AVG(avg_est_revenue_per_trip), 2)     AS AVG_EST_REVENUE_PER_TRIP,
    SUM(total_aborts)                       AS total_aborts
FROM segmented
GROUP BY segment
ORDER BY segment;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
    WITH segmented AS (
        SELECT
            driver_id,
            CASE WHEN AVG(DRIVER_ABORT_RATE) * 100 >= 30 
                 THEN 'High Abort' 
                 ELSE 'Reliable' 
            END                              AS segment,
            AVG(BOOKING_COMPLETION_RATE)*100 AS avg_completion_rate,
            AVG(AVG_est_REVENUE_PER_TRIP)        AS avg_est_revenue_per_trip,
            SUM(driver_abort_count)          AS total_aborts
        FROM ABC_ANALYTICS.MARTS.AGG_DRIVER_ACTIVITY
        WHERE total_bookings > 0
        GROUP BY driver_id
    )
    SELECT
        segment,
        COUNT(DISTINCT driver_id)           AS driver_count,
        ROUND(AVG(avg_completion_rate), 1)  AS avg_completion_rate,
        ROUND(AVG(avg_est_revenue_per_trip), 2) AS avg_est_revenue_per_trip,
        SUM(total_aborts)                   AS total_aborts
    FROM segmented
    GROUP BY segment
    ORDER BY segment
""").to_pandas()

df.columns = df.columns.str.lower()


colors = ['#EF4444' if s == 'High Abort' else '#10B981'
          for s in df['segment']]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('High Abort Drivers — Impact on Platform', fontweight='bold')

def style(ax, title, ylabel=''):
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

axes[0].bar(df['segment'], df['driver_count'], color=colors, width=0.4)
axes[0].bar_label(axes[0].containers[0], fontsize=9, fontweight='bold')
style(axes[0], 'Driver Count', 'Drivers')

axes[1].bar(df['segment'], df['avg_completion_rate'], color=colors, width=0.4)
axes[1].bar_label(axes[1].containers[0], fmt='%.1f%%', fontsize=9, fontweight='bold')
style(axes[1], 'Avg Completion Rate', '%')

axes[2].bar(df['segment'], df['avg_est_revenue_per_trip'], color=colors, width=0.4)
axes[2].bar_label(axes[2].containers[0], fmt='€%.2f', fontsize=9, fontweight='bold')
style(axes[2], 'Avg Revenue per Trip', 'EUR')

plt.tight_layout()
plt.show()

In [ ]:
%%sql -r dataframe_3
 WITH segmented AS (
        SELECT
            driver_id,
            CASE WHEN AVG(DRIVER_ABORT_RATE) * 100 >= 30 
                 THEN 'High Abort' 
                 ELSE 'Reliable' 
            END                              AS segment,
            AVG(BOOKING_COMPLETION_RATE)*100 AS avg_completion_rate,
            AVG(AVG_est_REVENUE_PER_TRIP)        AS avg_est_revenue_per_trip,
            SUM(driver_abort_count)          AS total_aborts
        FROM ABC_ANALYTICS.MARTS.AGG_DRIVER_ACTIVITY
        WHERE total_bookings > 0
        GROUP BY driver_id
    )

    select count(s.driver_id) as global_driver_count, count(d.driver_id) as dim_driver_count
    ,s.segment, avg(d.driver_rating)
    from segmented s
    left join marts.vw_dim_drivers d
    on s.driver_id=d.driver_id
    group by s.segment;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
    SELECT
        a.activity_date,
        d.country_name,
        ROUND(AVG(a.DRIVER_ABORT_RATE * 100), 1)       AS avg_abort_rate,
        ROUND(AVG(a.BOOKING_COMPLETION_RATE * 100), 1)  AS avg_completion_rate,
        ROUND(AVG(a.AVG_est_REVENUE_PER_TRIP), 2)           AS avg_est_revenue_per_trip
    FROM ABC_ANALYTICS.MARTS.AGG_DRIVER_ACTIVITY a
    LEFT JOIN ABC_ANALYTICS.MARTS.vw_DIM_DRIVERS d ON a.driver_id = d.driver_id
    WHERE a.total_bookings > 0
    GROUP BY a.activity_date, d.country_name
    ORDER BY a.activity_date
""").to_pandas()

df.columns = df.columns.str.lower()
df['activity_date'] = pd.to_datetime(df['activity_date'])

# split by country
fr = df[df['country_name'] == 'France']
at = df[df['country_name'] == 'Austria']

# ── Plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Driver Engagement Trends by Country — 01 to 14 Jun 2021',
             fontsize=12, fontweight='bold')

metrics = [
    ('avg_abort_rate',       'Abort Rate %',            '%'),
    ('avg_completion_rate',  'Booking Completion Rate %','%'),
    ('avg_est_revenue_per_trip', 'Avg Revenue per Trip',    '€'),
]

for ax, (col, title, unit) in zip(axes, metrics):
    ax.plot(fr['activity_date'], fr[col],
            color='#0EA5E9', linewidth=2, marker='o', markersize=5, label='France')
    ax.plot(at['activity_date'], at[col],
            color='#F59E0B', linewidth=2, marker='o', markersize=5, label='Austria')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel(unit, fontsize=9)
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

df = session.sql("""
    SELECT
        activity_date,
        ROUND(AVG(DRIVER_ABORT_RATE * 100), 1)       AS avg_abort_rate,
        ROUND(AVG(BOOKING_COMPLETION_RATE * 100), 1)  AS avg_completion_rate,
        ROUND(AVG(AVG_est_REVENUE_PER_TRIP), 2)           AS avg_revenue_per_trip
    FROM ABC_ANALYTICS.MARTS.AGG_DRIVER_ACTIVITY
    WHERE total_bookings > 0
    GROUP BY activity_date
    ORDER BY activity_date
""").to_pandas()

df.columns = df.columns.str.lower()
df['activity_date'] = pd.to_datetime(df['activity_date'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Driver Engagement KPI Trends — 01 to 14 Jun 2021',
             fontsize=12, fontweight='bold')

metrics = [
    ('avg_abort_rate',       'Abort Rate %'),
    ('avg_completion_rate',  'Completion Rate %'),
    ('avg_revenue_per_trip', 'Avg Revenue per Trip (€)'),
]

for ax, (col, title) in zip(axes, metrics):
    ax.plot(df['activity_date'], df[$], linewidth=2, marker='o', markersize=5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('kpi_trends.png', dpi=150, bbox_inches='tight')
plt.show()